Create Dummy DataFrames

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as _sum, month, year, current_date, add_months

spark = SparkSession.builder.appName("Top5AdidasProducts").getOrCreate()

# Sales Data
sales_data = [
    (1, 101, "2026-02-10", 2),
    (2, 102, "2026-02-15", 5),
    (3, 101, "2026-02-20", 3),
    (4, 103, "2026-02-25", 7),
    (5, 104, "2026-02-05", 1),
    (6, 105, "2026-02-18", 6),
    (7, 101, "2026-01-10", 4)  # Old data (should be excluded)
]

sales_df = spark.createDataFrame(sales_data, ["sale_id", "product_id", "sale_date", "quantity"])

# Product Data
products_data = [
    (101, "Adidas Shoes", "Adidas"),
    (102, "Adidas T-shirt", "Adidas"),
    (103, "Adidas Jacket", "Adidas"),
    (104, "Nike Shoes", "Nike"),
    (105, "Adidas Shorts", "Adidas")
]

products_df = spark.createDataFrame(products_data, ["product_id", "product_name", "brand"])

Filter Last Month’s Data + Adidas Products

In [0]:
from pyspark.sql.functions import to_date

sales_df = sales_df.withColumn("sale_date", to_date(col("sale_date")))

last_month = add_months(current_date(), -1)

filtered_df = sales_df \
    .filter((month(col("sale_date")) == month(last_month)) & 
            (year(col("sale_date")) == year(last_month))) \
    .join(products_df, "product_id") \
    .filter(col("brand") == "Adidas")

Aggregate & Get Top 5

In [0]:
top_5_products = filtered_df \
    .groupBy("product_id", "product_name") \
    .agg(_sum("quantity").alias("total_quantity")) \
    .orderBy(col("total_quantity").desc()) \
    .limit(5)

top_5_products.show()